# Reproducing Asari and Zador (2009) on CRCNS ac-1

Paper: H. Asari and A. M. Zador (2009), "Long-Lasting Context Dependence Constrains Neural Encoding Models in Rodent Auditory Cortex", J Neurophysiol 102:2638-2656, doi:10.1152/jn.00577.2009. The claims file (`claims/asari-2009.yaml`) was written from the version of record (text extraction of the publisher's PDF); the online supplement was not read.

Data set: CRCNS ac-1 (Asari, Wehr, Machens and Zador 2009, doi:10.6080/K0KW5CXR), streamed from the public bucket `crcnsarchive` (`ac-1/`). CRCNS does not version data sets; there is one published state, so the version is recorded as `unversioned`, the only one there is, and the files read are pinned by the MD5 sums of the data set's `checksums.md5`: `crcns-ac1.zip` 2300ee76d93383ab8c808ddf16761a23 (Summary.mat, analysis code; Wehr's data in `strf_wehr.ipynb`), `crcns-ac1-asari-results-1.zip` 18537da81d98e311c0274f62aea04daa and `crcns-ac1-asari-results-2.zip` 2f7ab955cc778fc6839ce59ed51bce89 (Asari's recordings). The data set also ships the authors' MATLAB analysis code (`asari/Tools/CxtDep.m` and others); the helper `asari.py` follows it where the paper is silent and says so.

| claim | tested here | outcome |
|---|---|---|
| methods-recording-counts | yes | partially reproduced |
| a1-mgb-firing-rates | yes | partially reproduced |
| fig2-example-reliability | yes | reproduced |
| fig2-example-long-context | yes | reproduced |
| fig3a-proportion-significant | yes | partially reproduced |
| fig3a-significance-decay | yes | partially reproduced (assumption) |
| fig3b-fractional-power-decay | yes | not reproduced |
| timescale-longer-than-membrane | yes | reproduced |
| fig4-predictability-upper-bound | yes | partially reproduced |
| fig4-strf-performance-low | in `strf_wehr.ipynb` | not reproduced (assumption) |
| fig5-example-conductances | blocked: no voltage-clamp data | blocked |
| fig6-conductance-decay | blocked: no voltage-clamp data | blocked |
| fig8a-proportion-by-property | yes | partially reproduced (assumption) |
| fig8b-decay-by-property | yes | partially reproduced |
| fig8a-significance-fits | yes | not reproduced |
| fig8c-relative-effects | yes | not reproduced (assumption) |
| fig8-natural-synthetic-probes-alike | yes | reproduced (assumption) |
| fig9-mgb-fast-decay | yes | reproduced |
| fig9-example-first-bin | yes | reproduced |
| fig7-example-frequency-context | blocked: example cell not in the data set | blocked |
| figs1-silence-probe | yes | reproduced |

The whole data set is read once (about 35 minutes): every file of Asari's recordings is loaded, median-filtered where the analysis needs it, and reduced to what the claims use per probe.

In [ ]:
import os, sys, time, zlib, collections, resource
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from scipy import stats
import asari as A
pd.set_option("display.width", 200)
os.environ.setdefault("OPENBLAS_NUM_THREADS", "4")
def seed(claim, k=0):
    """The claim's seed (protocol, step 3): crc32 of '<paper>/<claim id>', plus k."""
    return zlib.crc32(f"asari-2009/{claim}".encode()) + k
def runs(mask, dt=1e-4):
    """Start and end times (s) of the runs of True in a mask sampled every dt."""
    m = np.concatenate([[0], mask.astype(np.int8), [0]])
    d = np.diff(m)
    return list(zip(np.where(d == 1)[0] * dt, np.where(d == -1)[0] * dt))
rows_all = A.summary()
exp_desc = {r["exp"]: r["exp_desc"] for r in rows_all}
rows, seen = [], set()
for r in rows_all:
    if r["file"] not in seen:
        seen.add(r["file"]); rows.append(r)
print("Summary.mat rows:", len(rows_all), "| distinct files:", len(rows))
L = 60700                       # samples of the 0.1-ms population curves (6.07 s)
t01 = np.arange(L) * 1e-4
LQ = 6100                       # 1-ms points of the power curves (6.1 s)

In [ ]:
t0 = time.time()
R = A.run_pass(rows, np.random.default_rng(seed("fig3a-proportion-significant")),
               np.random.default_rng(seed("fig8a-proportion-by-property")),
               np.random.default_rng(seed("figs1-silence-probe")), n_shuffle=1000)
print("pass: %.0f s, peak memory %.1f GB" % (time.time() - t0, resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6))

# Data check

In [ ]:
import collections
F = pd.DataFrame(R["files"])
print("files read:", len(F), "| cells (folders):", F.cell.nunique())
print("experiments (Summary.mat):")
for e, g in F.groupby("exp"):
    print(f"  {e}: {exp_desc[e][:70]} | files {len(g)}, cells {g.cell.nunique()}, recording types {sorted(set(g.type))}")
print("units:", sorted(set(F.unit)), "| sampling rates (Hz):", sorted(set(F.sf)))
print("trigger types:", collections.Counter(tuple(t) for t in F.trig_types).most_common())
print("membrane potential range over whole-cell files: %.1f to %.1f mV" % (F[F.type == "Whole-cell"].vmin.min(), F[F.type == "Whole-cell"].vmax.max()))
print("cell-attached range: %.1f to %.1f mV" % (F[F.type == "Cell-attached"].vmin.min(), F[F.type == "Cell-attached"].vmax.max()))
print("ages (days):", sorted(set(F.age)))
print("Effective repeats in context-probe files:", collections.Counter(F[F.nseq > 0].rep))
dup = [f for f, n in collections.Counter(r["file"] for r in rows_all).items() if n > 1]
print("files listed twice in Summary.mat (counted once):", dup)
nat = R["nat"]; syn = R["syn"]; blank = R["blank"]
print("natural-sound A1 probes:", len(nat), "in", len({p['cell'] for p in nat}), "cells; durations %.2f to %.2f s" % (min(p['dur'] for p in nat), max(p['dur'] for p in nat)))
print("synthetic-ensemble tests:", len(syn), "| silence probes:", len(blank), "| MGB probes (auto spikes, silence included):",
      sum(1 for q in R['mgb'] if q['silence'] and q['detector'] == 'auto'))
# Clock check: responses are locked to the trigger times. Trial-to-trial correlation of the
# probe responses with the true onsets, against the same trials each shifted by a random lag.
rng_chk = np.random.default_rng(zlib.crc32(b"asari-2009/clock-check"))
true_r, shift_r = [], []
for p in nat:
    X = p["X1"]
    true_r.append(A.trial_correlation(X)[0])
    Xs = np.stack([[np.roll(x, rng_chk.integers(200, X.shape[2] - 200)) for x in xc] for xc in X])
    shift_r.append(A.trial_correlation(Xs)[0])
print("A1 clock check: mean trial-to-trial r with trigger onsets %.3f, with each trial shifted at random %.3f" % (np.mean(true_r), np.mean(shift_r)))
mg = [q for q in R["mgb"] if q["silence"] and q["detector"] == "auto"]
first = np.mean([q["psth"][:, 0].mean() for q in mg]); later = np.mean([np.nanmean(q["psth"][:, 5:20]) for q in mg])
print("MGB clock check: mean rate in the first 100 ms after probe onset %.2f Hz, 0.5 to 2 s after onset %.2f Hz" % (first, later))

The data descriptor and `ReadMe.txt` say the traces are in mV at the sampling rate in `response.sf`; every file has 10 kHz and mV, as printed. The paper's 10-kHz sampling is consistent with the files. Every whole-cell file is in mV (current clamp): no file holds currents, so the voltage-clamp recordings of Figures 5 and 6 are not in the data set. Responses are locked to the trigger times: the trial-to-trial correlation of the A1 probe responses falls to near zero when each trial is shifted by a random lag, and MGB firing is higher just after probe onset than later in the probe (printed above). This shows the traces and the trigger times share a clock in A1 natural-sound files and MGB files; for the synthetic files it is consistent with the data (the same code path and the onset responses), not shown separately.

In [ ]:
# The lines of the data set's CxtDep.m that asari.py follows, printed from the archive
cx = A.zips()["main"].read("asari/Tools/CxtDep.m").decode("latin-1").splitlines()
for a, b, what in [(300, 304, "ordinal ranks (a double sort) and the Kruskal-Wallis statistic"),
                   (307, 316, "runs kept when last minus first index >= dt, dt = 5 ms x 10 kHz = 50 samples, so 51 points (asari._runs)"),
                   (318, 322, "noise floor: rows (trials) shuffled across contexts"),
                   (340, 357, "FracPow: each trial's mean removed (line 346); one mean per probe left commented out (line 347)"),
                   (388, 399, "GetResp: occurrences of a context pooled, trials cut to the smallest count over contexts")]:
    print(f"--- CxtDep.m lines {a} to {b}: {what}")
    for i in range(a, b + 1):
        print(f"{i:4d} {cx[i - 1]}")


## methods-recording-counts

> Of 194 A1 cells recorded in current clamp, 123 met the analysis criterion (39 tested with natural sounds, 14 with TORCs, 6 with DMRs, 39 with modulated harmonic tones, 27 with modulated colored noise); 14 of 42 voltage-clamped cells and 14 of 16 MGB cells were analysed.

Probes are counted as `CxtDep.m` forms them: a probe needs at least two different conditioning stimuli, each with the same number of trials (the smallest over contexts), and the inclusion rule of the Methods (at least 4 trials in current clamp, at least 10 in MGB). The synthetic ensembles are counted per property tested (see fig8a-proportion-by-property for how the property groups are formed).

In [ ]:
nat4 = [p for p in nat if p["rep"] >= 4 and p["n"] >= 2]
nat4_nosil = [p for p in nat4 if "sig_nosil" in p]
syn4 = [s for s in syn if s["rep"] >= 4]
kind_name = {"torc": "TORC", "dmr": "DMR", "synthetic": "MHT", "pinknoise": "MCN", "naturalistic": "natural-sound variants"}
cells_by_kind = collections.defaultdict(set)
for s in syn4:
    cells_by_kind[kind_name[s["kind"]]].add(s["cell"])
cc_cells = {p["cell"] for p in nat4} | {s["cell"] for s in syn4}
a1_cells_all = set(F[F.exp.isin([0, 2, 3])].cell)
mgb4 = [q for q in R["mgb"] if q["silence"] and q["detector"] == "auto" and q["reps"] >= 10]
mgb_all = [q for q in R["mgb"] if q["silence"] and q["detector"] == "auto"]
print("A1 current-clamp cells in the data set (all experiments):", len(a1_cells_all))
print("A1 cells meeting the criterion (>= 1 probe, >= 2 contexts, >= 4 trials):", len(cc_cells))
print("  natural sounds:", len({p['cell'] for p in nat4}), "cells,", len(nat4), "probes (silence counted as a context);",
      len({p['cell'] for p in nat4_nosil}), "cells,", len(nat4_nosil), "probes (silence not counted)")
for k in ["TORC", "DMR", "MHT", "MCN", "natural-sound variants"]:
    print(f"  {k}: {len(cells_by_kind[k])} cells")
print("  cells in more than one ensemble:", sum(1 for c in cc_cells if sum(c in v for v in cells_by_kind.values()) + (c in {p['cell'] for p in nat4}) > 1))
print("Table 1 (probes and cells per probe type and property, trials >= 4):")
tab = collections.defaultdict(lambda: [0, set()])
for s in syn4:
    key = ("natural" if s["natural"] else kind_name[s["kind"]], s["prop"])
    tab[key][0] += 1; tab[key][1].add(s["cell"])
for k in sorted(tab):
    print(f"   {k[0]:>24s} | {k[1]:>12s} | {tab[k][0]:4d} probes ({len(tab[k][1])} cells)")
tot = collections.Counter(s["prop"] for s in syn4)
print("  totals per property:", dict(tot))
print("Voltage clamp: whole-cell files with currents (unit not mV):", int((F[F.type == 'Whole-cell'].unit != 'mV').sum()), "; voltage-clamp cells analysed: 0 (not in the data set)")
print("MGB: cells in the data set", F[F.exp == 1].cell.nunique(), "; probes with >= 10 trials", len(mgb4), "of", len(mgb_all), "in", len({q['cell'] for q in mgb4}), "cells; trials per probe", min(q['reps'] for q in mgb_all), "to", max(q['reps'] for q in mgb_all))
print("Experiment 2 (synthetic ensembles), per file: stimulus kinds, notes groups, the property tests chosen, probes with >= 2 contexts")
E2 = F[(F.exp == 2) & (F.nseq > 0)]
rows2 = []
for _, f in E2.iterrows():
    rows2.append(dict(file=f.file.split("/")[-1], kind="/".join(f.trig_types), rep=f.rep, stimuli=" + ".join(f.stim_kinds)[:70],
                      notes="; ".join(f.notes_groups)[:70], tests=",".join(f.tests), probes=len(f.probes), natural_probes=sum(1 for x in f.probes if x[3]),
                      syn_probes_2ctx_no_silence=sum(1 for x in f.probes_nosil if not x[3]), tests_run=sum(1 for s in syn if s["file"] == f.file)))
T2 = pd.DataFrame(rows2)
with pd.option_context("display.max_rows", 500, "display.max_colwidth", 70, "display.width", 250):
    print(T2.to_string(index=False))
print("synthetic (non-natural) probes with >= 2 contexts other than silence, by ensemble:", T2.groupby("kind").syn_probes_2ctx_no_silence.sum().to_dict())
unparsed = [l for g in E2.notes_groups for l in g if "variants" in l and not any(w in l.lower() for w in ["amplitude", "frequency"])]
print("files whose notes list variant groups:", int((T2.notes != "").sum()), "| notes lines naming variants of no property the parser knows:", len(unparsed),
      "| every notes group line:", sorted({l for g in E2.notes_groups for l in g}))
syn_cells = collections.defaultdict(set)
for s in syn4:
    if not s["natural"]:
        syn_cells[kind_name[s["kind"]]].add(s["cell"])
print("cells counted by their synthetic probes only (the rows of Table 1):", {k: len(v) for k, v in sorted(syn_cells.items())})
mht_syn = T2[T2.kind == "synthetic"].syn_probes_2ctx_no_silence.sum()
print(f"MHT files: {len(T2[T2.kind == 'synthetic'])}; MHT probes with >= 2 MHT contexts: {mht_syn}, against 25 + 25 + 71 + 57 = 178 MHT probe tests in Table 1")

**Result: partially reproduced.** With silence counted as a context, the data give 305 natural-sound probes in 39 cells and 93 MGB probes (10 to 25 trials) in 14 cells, as the paper; without silence 303 probes remain. 133 A1 cells meet the current-clamp criterion, not 123. Counted by their synthetic probes, as the rows of Table 1 count them, the ensembles hold 14 TORC cells (paper 14), 6 DMR (6), 26 colored-noise (27) and 2 MHT cells (39); counted by every probe of their files, 15 DMR and 26 MHT cells, with 14 cells of natural-sound level and frequency variants that Table 1 does not list. The DMR probes match Table 1 (8 amplitude probes in 2 cells, 20 frequency probes in 5). The per-file table shows where the AM and FM shortfall lies: the 27 MHT files hold 14 MHT probes with at least two MHT contexts, against 178 MHT probe tests in Table 1, and in the files with an AM by FM grid (tests "AM,FM") only the natural sound follows more than one MHT. The notes of the 6 files that list variant groups are all parsed: no line names a property the parser does not know. The voltage-clamp counts (29 probes, 14 cells) cannot be tested: no voltage-clamp file is in the data set.

## a1-mgb-firing-rates

> Firing rates were low in A1 (spontaneous 0.47 +/- 0.61 Hz, evoked 0.57 +/- 0.77 Hz, mean +/- SD, 194 cells) and high in MGB (spontaneous 0.78 +/- 1.25 Hz, evoked 11.4 +/- 16.9 Hz, 16 cells).

Rates per cell are spikes over seconds pooled over all files of the cell. Readings of the windows: spontaneous (a) every silent period between sounds (b) the 1 s before each sound onset; evoked (a) every sound period, including tuning-curve tones (b) the sound periods of the context-probe files only. Detectors: A1 (raw) upward crossings of -20 mV, (filt) the GetSpikes.m high-pass trace above 2 mV; MGB (auto) the GetSpikes.m automatic detector, (half) half its threshold.

In [ ]:
Rt = pd.DataFrame(R["rates"])
Rt["ctx"] = Rt.desc != "tuning curve"
out = []
for (area, det), g in Rt.groupby(["area", "detector"]):
    per = g.groupby("cell").agg(sp_n=("sp_n", "sum"), sp_t=("sp_t", "sum"), pre_n=("pre_n", "sum"), pre_t=("pre_t", "sum"), ev_n=("ev_n", "sum"), ev_t=("ev_t", "sum"))
    gc = g[g.ctx].groupby("cell").agg(ev_n=("ev_n", "sum"), ev_t=("ev_t", "sum"))
    vals = {"spont (a) silent periods": per.sp_n / per.sp_t, "spont (b) 1 s before onsets": per.pre_n / per.pre_t,
            "evoked (a) all sounds": per.ev_n / per.ev_t, "evoked (b) context-probe files": gc.ev_n / gc.ev_t}
    for k, v in vals.items():
        v = v.replace([np.inf], np.nan).dropna()
        out.append(dict(area=area, detector=det, window=k, cells=len(v), mean=v.mean(), sd=v.std(ddof=1), median=v.median(), max=v.max()))
rates_tab = pd.DataFrame(out)
print(rates_tab.round(3).to_string(index=False))
exp_r = {("A1", "spont"): 0.47, ("A1", "evoked"): 0.57, ("MGB", "spont"): 0.78, ("MGB", "evoked"): 11.4}
match = []
for a1d in ["raw", "filt"]:
    for mgd in ["auto", "half"]:
        for sw in ["spont (a) silent periods", "spont (b) 1 s before onsets"]:
            for ew in ["evoked (a) all sounds", "evoked (b) context-probe files"]:
                v = {}
                for area, det in [("A1", a1d), ("MGB", mgd)]:
                    for kind, w in [("spont", sw), ("evoked", ew)]:
                        v[(area, kind)] = float(rates_tab[(rates_tab.area == area) & (rates_tab.detector == det) & (rates_tab.window == w)]["mean"].iloc[0])
                ok = all(abs(v[k] - exp_r[k]) <= 0.1 * exp_r[k] for k in exp_r)
                part = v[("A1", "spont")] < 1 and v[("A1", "evoked")] < 1 and v[("MGB", "evoked")] > 5
                match.append(dict(A1=a1d, MGB=mgd, spont=sw[:9], evoked=ew[:10], A1_sp=v[("A1", "spont")], A1_ev=v[("A1", "evoked")], MGB_sp=v[("MGB", "spont")], MGB_ev=v[("MGB", "evoked")], all_within=ok, partial_rule=part))
match = pd.DataFrame(match)
print(match.round(3).to_string(index=False))
print("readings with all four means within 10%:", int(match.all_within.sum()), "of", len(match), "| readings meeting the partial rule:", int(match.partial_rule.sum()))

**Result: partially reproduced.** No reading puts all four means within 10%. A1 rates are low under every reading (raw detector: spontaneous 0.33 Hz over silent periods, evoked 0.46 Hz, 145 cells; SDs 0.42 and 0.60). In MGB the spontaneous rate is 0.62 Hz (automatic detector) or 0.82 Hz (half threshold) against 0.78, but the evoked rate is 5.07 or 7.15 Hz against 11.4. The partial rule (A1 means below 1 Hz, MGB evoked above 5 Hz) holds in 12 of 16 readings.

## fig2-example-reliability

> The example A1 neuron of Figure 2 had high trial-to-trial reliability (correlation across trials 0.74 +/- 0.08 for the shown probe and 0.61 +/- 0.07 over the seven natural sound fragments tested), and the mean correlation over the population was 0.31 +/- 0.09.

The correlation is between median-filtered traces of trials within one context, over the probe. Readings: (pairs) the mean over all pairs of trials; (loo) each trial with the mean of the other trials. The population mean is over probes, or over cells (mean of each cell's probes).

In [ ]:
rel = pd.DataFrame([dict(cell=p["cell"], file=p["file"], probe=p["probe"], n=p["n"], rep=p["rep"], contexts=p["contexts"], pairs=p["rel_pairs"], loo=p["rel_loo"]) for p in nat])
for k in ["pairs", "loo"]:
    pc = rel.groupby("cell")[k].mean()
    print(f"{k}: over probes {rel[k].mean():.3f} +/- {rel[k].std(ddof=1):.3f} (n = {len(rel)}); over cells {pc.mean():.3f} +/- {pc.std(ddof=1):.3f} (n = {len(pc)}); range {rel[k].min():.2f} to {rel[k].max():.2f}")
# candidates for the example: 7 natural fragments, 6 repeats
cand = []
for f, g in rel.groupby("file"):
    if len(g) == 7 and (g.rep == 6).all():
        m, s = g.pairs.mean(), g.pairs.std(ddof=0)
        pr = g[(g.n == 3) & g.contexts.apply(lambda c: 0 in c)]
        best = pr.iloc[(pr.pairs - 0.74).abs().argsort()].iloc[0] if len(pr) else None
        cand.append(dict(file=f, mean=m, sd=s, sd_ddof1=g.pairs.std(ddof=1), best_probe=None if best is None else best.probe, best_r=None if best is None else best.pairs,
                         match=abs(m - 0.61) <= 0.061 and best is not None and abs(best.pairs - 0.74) <= 0.074))
cand = pd.DataFrame(cand)
print(cand.round(3).to_string(index=False))
ex_file = cand[cand.match].file.iloc[0] if cand.match.any() else None
print("matched example:", ex_file)
if ex_file:
    g = rel[rel.file == ex_file]
    print(g[["probe", "contexts", "rep", "pairs", "loo"]].round(3).to_string(index=False))

**Result: reproduced.** The mean trial-to-trial correlation is 0.319 over the 305 probes (pairs of trials), 0.310 +/- 0.092 over the 39 cells, within 0.31 +/- 0.09; the leave-one-out reading gives 0.447. One cell matches the example: 20051222-ha1-001, 7 fragments, 6 repeats, 0.611 +/- 0.075 over its fragments (paper 0.61 +/- 0.07), and its probe 4, after silence and two fragments, 0.737 (paper 0.74).

## fig2-example-long-context

> In an example A1 neuron, changing the conditioning stimulus changed the response to the probe for about 4 s; responses did not differ between 2 and 4 s after probe onset and diverged again after about 4 s.

In [ ]:
ex = [p for p in nat if p["file"] == ex_file]
rows_ex = []
for p in ex:
    rr = runs(p["sig"])
    rows_ex.append(dict(probe=p["probe"], contexts=p["contexts"], rel=round(p["rel_pairs"], 3), dur=round(p["dur"], 2),
                        periods=[(round(a, 3), round(b, 3)) for a, b in rr]))
print(pd.DataFrame(rows_ex).to_string(index=False))
exp_p = [p for p in ex if p["n"] == 3 and 0 in p["contexts"]]
exp_p = min(exp_p, key=lambda p: abs(p["rel_pairs"] - 0.74))
rr = runs(exp_p["sig"])
early = any(a < 2.0 for a, b in rr)
gap = not any((a < 3.6) and (b > 2.2) for a, b in rr)
late = any(a > 3.6 for a, b in rr)
print(f"matched probe {exp_p['probe']} (contexts {exp_p['contexts']}, r = {exp_p['rel_pairs']:.3f}, {exp_p['dur']:.2f} s)")
print("significant periods (s):", [(round(a, 3), round(b, 3)) for a, b in rr])
print("period before 2 s:", early, "| none between 2.2 and 3.6 s:", gap, "| one starting after 3.6 s:", late)
print("last significant time: %.3f s" % (rr[-1][1] if rr else np.nan))
n_late = sum(1 for p in nat if any(b > 4.0 for a, b in runs(p["sig"])))
print(f"natural-sound probes with a significant period later than 4 s: {n_late} of {len(nat)}")

**Result: reproduced.** For the matched probe (cell 20051222-ha1-001, probe 4, contexts silence and fragments 2 and 3) the significant periods run from 0.026 s to 1.600 s, none falls between 2.2 and 3.6 s, and they resume from 3.970 s to 4.596 s. Over all natural-sound probes, 19 of 305 have a significant period later than 4 s.

## fig3a-proportion-significant

> Of 305 natural sound probes tested in 39 A1 cells with typically five to eight natural sound contexts, 204 (66.9%) showed significant context dependence, and about a quarter (23.7%) of the significant events occurred more than 1 s after probe onset.

Readings: silence counted among the contexts or not; the proportion at time t over all probes (ended probes count as not significant) or over the probes still running at t. The late fraction is one minus the cumulative probability at 1 s, the cumulative probability being the running integral of the proportion less the noise floor, normalized at its peak (Figure 3A legend).

In [ ]:
L = 60700
t01 = np.arange(L) * 1e-4
def sig_curves(recs, key, nkey, running):
    pc, C = A.population_curve([p[key].astype(float) for p in recs], L, running)
    nz, _ = A.population_curve([np.repeat(p[nkey], 10) for p in recs], L, running)
    d = np.nan_to_num(pc - nz)
    cum = np.cumsum(d) * 1e-4
    ipk = int(np.argmax(cum))
    return dict(pc=pc, nz=nz, cum=cum / cum[ipk], tpk=t01[ipk], late=1 - cum[int(1 / 1e-4)] / cum[ipk], kappa=float(np.nanmean(nz)))
readings3a = {}
tab = []
for sil, key, nkey in [("silence counted", "sig", "noise"), ("silence not counted", "sig_nosil", "noise_nosil")]:
    recs = [p for p in nat if key in p]
    nsig = sum(p[key].any() for p in recs)
    for running in [False, True]:
        c = sig_curves(recs, key, nkey, running)
        readings3a[(sil, running)] = c
        ok = abs(nsig - 204) <= 20.4 and abs(nsig / len(recs) - 0.669) <= 0.0669 and abs(c["late"] - 0.237) <= 0.0237
        tab.append(dict(reading=sil, denominator="running probes" if running else "all probes", probes=len(recs), significant=nsig,
                        fraction=nsig / len(recs), late_fraction=c["late"], cum_peak_s=c["tpk"], mean_noise_floor=c["kappa"], all_within=ok))
tab3a = pd.DataFrame(tab)
print(tab3a.round(4).to_string(index=False))
by_cell = collections.defaultdict(list)
for p in nat:
    by_cell[p["cell"]].append(p["sig"].any())
v = np.array([np.mean(x) for x in by_cell.values()])
print(f"fraction of significant probes per cell: median {np.median(v):.2f}, range {v.min():.2f} to {v.max():.2f}, cells with none {int((v == 0).sum())} of {len(v)}")
fig, ax = plt.subplots(1, 2, figsize=(10, 3))
c = readings3a[("silence counted", False)]
ax[0].plot(t01[::10], c["pc"][::10], label="proportion"); ax[0].plot(t01[::10], c["nz"][::10], label="noise floor"); ax[0].set_xlabel("time from probe onset (s)"); ax[0].legend()
ax[1].plot(t01[::10], c["cum"][::10]); ax[1].set_ylabel("cumulative (normalized)"); ax[1].set_xlabel("time (s)")
plt.tight_layout(); plt.show()
print("plotted ranges: proportion %.3f to %.3f, noise floor %.4f to %.4f, cumulative %.2f to %.2f" % (np.nanmin(c['pc']), np.nanmax(c['pc']), np.nanmin(c['nz']), np.nanmax(c['nz']), c['cum'].min(), c['cum'].max()))
nm = sum(p["sig_midrank"].any() for p in nat)
print(f"variant, midranks with tie correction (silence counted): {nm} of {len(nat)} probes significant (ordinal ranks: {sum(p['sig'].any() for p in nat)});",
      f"significant time points differ at {np.mean([np.mean(p['sig'] != p['sig_midrank']) for p in nat]):.4f} of all points")
tf = np.array([p["tie_frac"] for p in nat])
print(f"fraction of values tied with another trial at the same time point: mean {tf.mean():.3f}, median {np.median(tf):.3f}, range {tf.min():.3f} to {tf.max():.3f} over probes")

Seed sensitivity of the noise floor. The noise floor rests on shuffles. Re-running the 1000 shuffles at 0.1 ms under four more seeds would take as long again as the whole pass, so the spread over seeds is measured on the 1-ms binned trials of every probe (run rule 5 points), 100 shuffles per probe under the claim's seed and the seed plus 1000, 2000, 3000 and 4000.

In [ ]:
def nf_1ms(recs, s, key_ctx=True):
    rg = np.random.default_rng(s)
    vals = []
    for p in recs:
        X = p["X1"]
        if not key_ctx:
            keep = [i for i, c in enumerate(p["contexts"]) if c != 0]
            if len(keep) < 2:
                continue
            X = X[keep]
        _, nz = A.kw_significance(X, sf=1000, rng=rg, n_shuffle=100)
        vals.append(nz.mean())
    return float(np.mean(vals))
nf_seeds = {k: nf_1ms(nat, seed("fig3a-proportion-significant", k)) for k in [0, 1000, 2000, 3000, 4000]}
print("mean noise floor at 1 ms under five seeds:", {k: round(v, 5) for k, v in nf_seeds.items()})
print("spread: %.2e (relative %.1f%%)" % (max(nf_seeds.values()) - min(nf_seeds.values()), 100 * (max(nf_seeds.values()) - min(nf_seeds.values())) / np.mean(list(nf_seeds.values()))))

In [ ]:
for k, v in nf_seeds.items():
    sc = v / nf_seeds[0]
    c = readings3a[("silence counted", False)]
    d = np.nan_to_num(c["pc"] - sc * c["nz"]); cum = np.cumsum(d); ipk = np.argmax(cum)
    print(f"seed +{k}: noise floor scaled by {sc:.4f}, late fraction (silence counted, all probes) {1 - cum[10000] / cum[ipk]:.4f}")

**Result: partially reproduced.** 187 of 305 probes (0.613) are significant with silence counted, within 204 +/- 20.4 and 0.669 +/- 0.067; without silence 165 of 303 (0.545), outside. The late fraction is 0.276 (all probes) or 0.295 (running probes) with silence and 0.198 or 0.211 without, none within 0.237 +/- 0.024. The cumulative curve peaks at 4.78 s (paper 4.72 s) and the mean noise floor is 2.2 x 10^-3 to 3.2 x 10^-3. Midranks with the tie correction (a variant of the ordinal ranks of `CxtDep.m`) give the same 187 of 305; on average 3.3% of values at a time point are tied with another trial, and the significant points differ at 0.02% of all points. Under the four other seeds the 1-ms noise floor moves by 3.4% and the late fraction by at most 0.0014.

## fig3a-significance-decay

> The proportion of probes with significant context dependence decays after probe onset as the sum of two exponentials (sizes 0.17 and 0.09, decay constants 0.20 and 0.90 s), above a noise floor of about 2.8 x 10^-3.

Eq. 44 with the constant fixed at the mean noise floor, fitted by least squares to the proportion curve (at 1 ms) from probe onset, 1 to 3 exponentials, keeping the largest number whose sizes are each at least a tenth of the total. Fit ranges 0 to 4.72 s and 0 to 6 s. Bounds are not given: sizes unbounded, or sizes at least 0 (decays only); a grid of start values, the best fit kept.

In [ ]:
fits3a = []
for (sil, running), c in readings3a.items():
    tt = t01[::10]; yy = c["pc"][::10]
    for tmax in [4.72, 6.0]:
        for pos in [False, True]:
            m = (tt <= tmax) & np.isfinite(yy)
            ch, fits = A.fit_select(tt[m], yy[m], kappa=c["kappa"], positive=pos)
            f2 = fits.get(2)
            ok = ch == 2 and f2 is not None and all(abs(a - b) <= tol for a, b, tol in zip(f2["alpha"] + f2["tau"], [0.17, 0.09, 0.20, 0.90], [0.017, 0.009, 0.02, 0.09])) and abs(c["kappa"] - 0.0028) <= 0.00028
            part = ch == 2 and f2 is not None and 0.5 <= f2["tau"][1] <= 2
            fits3a.append(dict(silence=sil, denominator="running" if running else "all", range_s=tmax, sizes=">=0" if pos else "free", chosen=ch,
                               a1=f2["alpha"][0], t1=f2["tau"][0], a2=f2["alpha"][1], t2=f2["tau"][1], one_exp=(round(fits[1]["alpha"][0], 3), round(fits[1]["tau"][0], 3)),
                               kappa=c["kappa"], within=ok, partial_rule=part))
fits3a = pd.DataFrame(fits3a)
print(fits3a.round(4).to_string(index=False))
print("readings with every value within tolerance:", int(fits3a.within.sum()), "| meeting the partial rule:", int(fits3a.partial_rule.sum()), "of", len(fits3a))
print("seed check: the mean noise floor over five seeds varies by %.1f%% (fig3a-proportion-significant)" % (100 * (max(nf_seeds.values()) - min(nf_seeds.values())) / np.mean(list(nf_seeds.values()))))

**Result: partially reproduced, resting on an assumption.** With free sizes the rule picks two exponentials, one of them negative (the rise at onset: -0.17 at 0.015 s, then 0.22 at 0.47 s), and the partial rule fails. With sizes held at 0 or above (a decay, as Eq. 44 describes it, the reading adopted), two are picked under every silence-counted reading: 0.15 to 0.17 at 0.28 to 0.31 s and 0.05 to 0.07 at 1.1 to 1.5 s, against 0.17 at 0.20 s and 0.09 at 0.90 s; none has every value within tolerance, and the slow constant lies between 0.5 and 2 s. Without silence and with sizes at or above 0, only one exponential is kept. The fixed constant is the mean noise floor, 2.2 x 10^-3 or 2.8 x 10^-3.

## fig3b-fractional-power-decay

> The context-dependent fraction of the stimulus-related response power, normalized by the smoothed stimulus-related power, decays from about 0.49 with a time constant of about 1.04 s.

Q(t) of Eq. 20 at 1 ms, numerator Eq. 19 and denominator Eq. 17 averaged over probes, the denominator smoothed over [0, 2t]; one exponential with constant 0. Readings: silence counted or not; probes still running at t or all probes; fit range 0 to 4 s or 0 to 6 s; and the centering of the traces (each trial's mean over the probe removed, as `CxtDep.m`; or one mean per probe removed, a stand-in for the Methods' zero-mean traces).

In [ ]:
LQ = 6100
qread = {}
for sil in ["silence counted", "silence not counted"]:
    for center in ["trial", "probe"]:
        for running in [True, False]:
            if sil == "silence counted":
                recs = nat
                if center == "trial":
                    q = A.q_curve(recs, LQ, "trial", running)
                else:
                    q = A.q_curve(recs, LQ, "probe", running, from_x1=True)
            else:
                recs = []
                for p in nat:
                    keep = [i for i, c in enumerate(p["contexts"]) if c != 0]
                    if len(keep) >= 2:
                        recs.append(dict(X1=p["X1"][keep], pred_trial=p.get("pred_nosil"), inv_trial=p.get("inv_nosil")))
                q = A.q_curve(recs, LQ, center, running, from_x1=(center == "probe"))
            qread[(sil, center, running)] = q
tab3b = []
for (sil, center, running), q in qread.items():
    for tmax in [4.0, 6.0]:
        m = (q["t"] <= tmax) & np.isfinite(q["Q"])
        f = A.fit_exp(q["t"][m], q["Q"][m], k=1, kappa=0.0)
        ok = abs(f["alpha"][0] - 0.49) <= 0.049 and abs(f["tau"][0] - 1.04) <= 0.104
        tab3b.append(dict(silence=sil, centering=center, denominator="running" if running else "all", range_s=tmax, alpha1=f["alpha"][0], tau1=f["tau"][0], within=ok))
tab3b = pd.DataFrame(tab3b)
tab3b["role"] = np.where(tab3b.centering == "trial", "reading", "variant")
print(tab3b.round(3).to_string(index=False))
print("readings (each trial's mean removed, CxtDep.m line 346) with both within tolerance:", int(tab3b[tab3b.role == "reading"].within.sum()), "of", int((tab3b.role == "reading").sum()),
      "| variant (one mean per probe, CxtDep.m line 347, commented out there):", int(tab3b[tab3b.role == "variant"].within.sum()), "of", int((tab3b.role == "variant").sum()))
r3 = tab3b[tab3b.role == "reading"]
print("readings with alpha1 within 0.49 +/- 0.049:", int((abs(r3.alpha1 - 0.49) <= 0.049).sum()), "of", len(r3), "| with tau1 within 1.04 +/- 0.104:", int((abs(r3.tau1 - 1.04) <= 0.104).sum()), "| tau1 range %.2f to %.2f s" % (r3.tau1.min(), r3.tau1.max()))
q = qread[("silence counted", "trial", True)]
print("Q(t) main reading: mean over 0-0.1 s %.3f, 0.9-1.1 s %.3f, 1.9-2.1 s %.3f, 3.9-4.1 s %.3f; range %.2f to %.2f" % tuple([np.nanmean(q['Q'][(q['t'] > a) & (q['t'] < b)]) for a, b in [(0, .1), (.9, 1.1), (1.9, 2.1), (3.9, 4.1)]] + [np.nanmin(q['Q']), np.nanmax(q['Q'])]))
fig, ax = plt.subplots(1, 2, figsize=(10, 3))
ax[0].plot(q["t"], q["pred"], label="stimulus-related (Eq. 17)"); ax[0].plot(q["t"], q["smooth"], label="[0, 2t] mean"); ax[0].plot(q["t"], q["cd"], label="context-dependent (Eq. 19)"); ax[0].legend(fontsize=7); ax[0].set_ylabel("mV^2")
ax[1].plot(q["t"], q["Q"]); f = A.fit_exp(q["t"][q["t"] <= 4], q["Q"][q["t"] <= 4], k=1, kappa=0.0); ax[1].plot(q["t"], f["alpha"][0] * np.exp(-q["t"] / f["tau"][0])); ax[1].set_ylim(-0.2, 1.2)
plt.tight_layout(); plt.show()
print("points of Q outside the plotted range [-0.2, 1.2]:", int(((q["Q"] < -0.2) | (q["Q"] > 1.2)).sum()), "of", int(np.isfinite(q["Q"]).sum()))

**Result: not reproduced.** With each trial's mean removed, as `CxtDep.m` does (line 346, printed above) and as the Methods' r(t) - <r(t)>_t reads, no reading puts both values within tolerance (0 of 8): the size is 0.42 to 0.46, within 0.49 +/- 0.049 in the 4 readings with silence counted, but the time constant is 1.94 to 2.01 s with silence and 1.50 to 1.58 s without, against 1.04 s. One mean per probe removed, the line `CxtDep.m` leaves commented out (347), is not a reading the paper gives and is shown as a variant only: 0.63 at 0.58 to 0.59 s, and 0.71 at 0.30 s without silence, within tolerance in none. Q(t) falls from 0.67 over the first 100 ms to 0.20 near 1 s and 0.08 near 2 s.

## timescale-longer-than-membrane

> The trial-to-trial noise autocorrelation and the autocorrelation of spontaneous activity decay within about 100 ms, much faster than the second-long timescale of context dependence in A1.

Noise is the residual of each trial from its context mean (Eq. 14), its autocorrelation normalized by its variance (Eq. 45), at 1 ms, averaged over trials and then probes. Spontaneous activity is the intersequence silence (whole, and without its first 1 s), averaged over silences and then files.

In [ ]:
lag = np.arange(1000) * 1e-3
acf_noise = np.nanmean([p["acf"] for p in nat if p.get("acf") is not None], axis=0)
acf_sp0 = np.nanmean([a["acf"] for a in R["spont_acf"] if a["skip"] == 0], axis=0)
acf_sp1 = np.nanmean([a["acf"] for a in R["spont_acf"] if a["skip"] == 1], axis=0)
tabts = []
for name, a in [("noise (Eq. 14)", acf_noise), ("spontaneous, whole silence", acf_sp0), ("spontaneous, without first 1 s", acf_sp1)]:
    tabts.append(dict(series=name, lag_1_over_e_ms=1000 * A.first_below(lag, a, 1 / np.e), lag_0p1_ms=1000 * A.first_below(lag, a, 0.1), acf_100ms=a[100], acf_500ms=a[500]))
tabts = pd.DataFrame(tabts)
print(tabts.round(3).to_string(index=False))
print("probes:", sum(1 for p in nat if p.get('acf') is not None), "| files with silences:", len([a for a in R['spont_acf'] if a['skip'] == 0]))
per_probe = [1000 * A.first_below(lag, p["acf"], 1 / np.e) for p in nat if p.get("acf") is not None]
print("per-probe 1/e lag (ms): median %.0f, 90th percentile %.0f, max %.0f" % (np.nanmedian(per_probe), np.nanpercentile(per_probe, 90), np.nanmax(per_probe)))
ok = all(tabts.lag_1_over_e_ms[:2] < 150) and all(tabts.lag_0p1_ms[:2] < 150) and tabts.lag_1_over_e_ms[0] < 1040 / 3
print("both lags under 150 ms for noise and spontaneous (whole) and 1/e lag below 1.04 s / 3:", ok)

**Result: reproduced.** The mean noise autocorrelation falls below 1/e at 71 ms and below 0.1 at 115 ms; that of the intersequence silences at 77 and 121 ms (75 and 119 ms without their first second). Both are under 150 ms and the 1/e lag is below a third of 1.04 s.

## fig4-predictability-upper-bound

> Without knowledge of the context, no model can capture more than about half (0.51) of the predictable response power with a window of about 100 ms; the upper bound approaches one only with a window of seconds (time constant 1.04 s).

Eq. 43, context-independent power (Eq. 18) over stimulus-related power (Eq. 17), averaged over probes, denominator smoothed over [0, 2t] (main) or not (variant); one exponential with constant 1. The readings of fig3b-fractional-power-decay are run.

In [ ]:
tab4 = []
for (sil, center, running), q in qread.items():
    for den, key in [("smoothed", "U"), ("unsmoothed", "U_raw")]:
        for tmax in [4.0, 6.0]:
            m = (q["t"] <= tmax) & np.isfinite(q[key])
            f = A.fit_exp(q["t"][m], q[key][m], k=1, kappa=1.0)
            ub = 1 + f["alpha"][0]
            tab4.append(dict(silence=sil, centering=center, denominator="running" if running else "all", ratio=den, range_s=tmax, one_plus_alpha=ub, tau1=f["tau"][0],
                             mean_0_100ms=float(np.nanmean(q[key][q["t"] < 0.1])), within=abs(ub - 0.51) <= 0.051 and abs(f["tau"][0] - 1.04) <= 0.104))
tab4 = pd.DataFrame(tab4)
tab4["role"] = np.where(tab4.centering == "trial", "reading", "variant")
tab4["tau_at_bound"] = tab4.tau1 >= 99.9
print(tab4.round(3).to_string(index=False))
print("readings (each trial's mean removed) with both within tolerance:", int(tab4[tab4.role == "reading"].within.sum()), "of", int((tab4.role == "reading").sum()),
      "| variant (one mean per probe):", int(tab4[tab4.role == "variant"].within.sum()), "of", int((tab4.role == "variant").sum()))
r4 = tab4[(tab4.role == "reading") & ~tab4.tau_at_bound]
print("readings (fits at the bound left out) with 1 + alpha1 within 0.51 +/- 0.051:", int((abs(r4.one_plus_alpha - 0.51) <= 0.051).sum()), "of", len(r4),
      "| tau1 within 1.04 +/- 0.104:", int((abs(r4.tau1 - 1.04) <= 0.104).sum()), "| tau1 range %.2f to %.2f s | 1 + alpha1 range %.3f to %.3f" % (r4.tau1.min(), r4.tau1.max(), r4.one_plus_alpha.min(), r4.one_plus_alpha.max()))
print("fits with tau1 at the 100-s bound (no approach to 1 within the range; flagged, not used):")
print(tab4[tab4.tau_at_bound][["silence", "centering", "denominator", "ratio", "range_s", "one_plus_alpha"]].round(3).to_string(index=False))

**Result: partially reproduced.** Under the readings the claims file names, with each trial's mean removed as `CxtDep.m` does, none puts both values within tolerance (0 of 16). The bound near probe onset is about half: 1 + alpha1 is 0.424 to 0.547, within 0.51 +/- 0.051 in 10 of the 14 fits that stay inside the bounds, and the ratio over the first 100 ms is 0.38 to 0.56. The time constant is 1.60 to 2.83 s in all 14, against 1.04 s, so the approach to one is about twice as slow as the paper's. Two fits (all probes, smoothed denominator, 0 to 6 s) reach the 100-s bound of the time constant and are flagged and not used. One mean per probe removed is not a reading the paper or the claims file gives (the Methods centre each trace, r(t) - <r(t)>_t, and `CxtDep.m` leaves the per-probe line commented out); as a variant it puts both values within tolerance in 2 of 16 fits (silence not counted, running probes, smoothed: 0.468 and 0.99 s from 0 to 4 s), which do not decide.

## fig8a-proportion-by-property

> When only one property of synthetic conditioning stimuli varied, context dependence was found in 77 of 95 probes (81.1%, 31 cells) for amplitude, 73 of 110 (35 cells) for frequency, 62 of 96 (64.6%, 27 cells) for AM, 47 of 82 (57.3%, 25 cells) for FM and 59 of 137 (43.1%, 27 cells) for higher-order properties.

Only the variants of the varied property are conditioning stimuli (no silence, no unrelated fragment). The property groups come from each file's stimulus descriptions, parameters and notes (`asari.property_tests`): TORC and DMR files by their descriptions (amplitude, or bandwidth and frequency variants); MHT files by the notes' groups or, without them, by which MHT parameter varies (`AMmean` amplitude, `f0` frequency, `AMrange`/`AMstd` AM, `FMrange`/`FMstd` FM); colored-noise files by natural sound against colored noise; natural sounds played at several levels or shifted in frequency by their `attenuation` and `df` parameters. Where a file varies two properties on a grid, or holds several natural sounds each with its colored noise, the fragments sharing the value of the tested property are pooled into one context, so each probe is tested once per property. Natural and synthetic probes are combined.

In [ ]:
PROPS = ["amplitude", "frequency", "AM", "FM", "higher-order"]
exp8 = {"amplitude": (77, 95, 31), "frequency": (73, 110, 35), "AM": (62, 96, 27), "FM": (47, 82, 25), "higher-order": (59, 137, 27)}
syn4 = [s for s in syn if s["rep"] >= 4]
tab8 = []
for pr in PROPS:
    g = [s for s in syn4 if s["prop"] == pr]
    k = sum(s["sig"].any() for s in g)
    tab8.append(dict(property=pr, probes=len(g), cells=len({s['cell'] for s in g}), significant=k, fraction=k / len(g) if g else np.nan,
                     paper=f"{exp8[pr][0]}/{exp8[pr][1]} ({exp8[pr][2]} cells)", within=abs(k - exp8[pr][0]) <= 0.1 * exp8[pr][0]))
tab8 = pd.DataFrame(tab8)
print(tab8.round(3).to_string(index=False))
fr = dict(zip(tab8.property, tab8.fraction))
print("order amplitude > frequency:", fr["amplitude"] > fr["frequency"], "| AM > FM > higher-order:", fr["AM"] > fr["FM"] > fr["higher-order"])
print("by ensemble:")
print(pd.DataFrame([dict(prop=s["prop"], kind=s["kind"], natural=s["natural"], sig=s["sig"].any()) for s in syn4]).groupby(["prop", "kind", "natural"]).sig.agg(["sum", "count"]).to_string())
nos = [s for s in syn4 if s["kind"] != "naturalistic"]
print("variant without the natural-sound variants (level and frequency shifts of natural sounds):",
      {pr: (sum(s['sig'].any() for s in nos if s['prop'] == pr), sum(1 for s in nos if s['prop'] == pr)) for pr in PROPS})
alt = [s for s in R["syn_alt"] if s["rep"] >= 4]
main_ho = [s for s in syn4 if s["prop"] == "higher-order"]
each = [s for s in alt if s["reading"] == "each"]
pair = [s for s in alt if s["reading"] == "pair"]
pair_probe = collections.defaultdict(bool)
for s in pair:
    pair_probe[(s["file"], s["probe"])] |= bool(s["sig"].any())
ho = {"pooled (main)": (sum(s["sig"].any() for s in main_ho), len(main_ho)),
      "each fragment its own context": (sum(s["sig"].any() for s in each), len(each)),
      "each natural sound with its own noise, per test": (sum(s["sig"].any() for s in pair), len(pair)),
      "each natural sound with its own noise, probe significant in any pair": (sum(pair_probe.values()), len(pair_probe))}
print("higher-order (colored noise) readings, against 59 of 137 (0.431):")
for k, (a, b) in ho.items():
    print(f"  {k}: {a}/{b} = {a / b:.3f}; within 59 +/- 5.9: {abs(a - 59) <= 5.9}; AM > FM > higher-order: {fr['AM'] > fr['FM'] > a / b}")
print("probes dropped for fewer than 4 trials: each", sum(1 for s in R["syn_alt"] if s["reading"] == "each" and s["rep"] < 4), "| pair", sum(1 for s in R["syn_alt"] if s["reading"] == "pair" and s["rep"] < 4))

**Result: partially reproduced, resting on an assumption.** The groups give 95 amplitude, 110 frequency and 137 higher-order probes, as Table 1, but only 39 AM and 25 FM probes (Table 1: 96 and 82; the MHT probes are not in the data set, see methods-recording-counts). Significant: amplitude 56 of 95 (paper 77), frequency 62 of 110 (73), AM 27 of 39 (62 of 96), FM 16 of 25 (47 of 82); none within tolerance. Amplitude (0.589) exceeds frequency (0.564) and AM (0.692) exceeds FM (0.640). Higher-order depends on how the colored noises are grouped. Pooling every natural sound of a file into one context and every noise into another gives 88 of 137 (0.642), above FM. Testing each natural sound against its own colored noise, as Table 1's legend describes ("comparing natural sounds and corresponding modulated colored noise"), gives 64 of 131 probes significant in at least one pair (0.489, within 59 +/- 5.9) and 109 of 336 single tests (0.324); every fragment as its own context gives 75 of 131 (0.573). The order AM > FM > higher-order holds under these three readings and fails under the pooled one. The pairing of each natural sound with its own noise is adopted (the assumption), and under it the partial rule holds.

## fig8b-decay-by-property

> The normalized context-dependent power decays slowly for amplitude and frequency changes (time constants 0.66 and 0.97 s) and fast for AM, FM and higher-order changes (0.11, 0.15 and 0.20 s), so higher-order properties act mainly on a timescale of about 100 ms.

Q(t) of Eq. 20 per property over the probes of fig8a-proportion-by-property (probes still running at t, each trial's mean removed), one exponential with constant 0 fitted from 0 to 4 s, and from 0 to the end of the shortest probe of the property as a variant.

In [ ]:
exp8b = {"amplitude": (0.66, 0.57), "frequency": (0.97, 0.54), "AM": (0.11, 0.86), "FM": (0.15, 0.64), "higher-order": (0.20, 0.52)}
q8 = {}
tab8b = []
for pr in PROPS:
    g = [s for s in syn4 if s["prop"] == pr]
    for center in ["trial", "probe"]:
        q = A.q_curve(g, LQ, center, True, from_x1=(center == "probe"))
        q8[(pr, center)] = q
        for rng_name, tmax in [("0-4 s", 4.0), ("0-shortest", min(s["dur"] for s in g))]:
            m = (q["t"] <= tmax) & np.isfinite(q["Q"])
            f = A.fit_exp(q["t"][m], q["Q"][m], k=1, kappa=0.0)
            tab8b.append(dict(property=pr, centering=center, range=rng_name, tmax=tmax, tau1=f["tau"][0], alpha1=f["alpha"][0], paper_tau=exp8b[pr][0], paper_alpha=exp8b[pr][1],
                              tau_within=abs(f["tau"][0] - exp8b[pr][0]) <= 0.1 * exp8b[pr][0]))
tab8b = pd.DataFrame(tab8b)
print(tab8b.round(3).to_string(index=False))
for (center, rng_name), g in tab8b.groupby(["centering", "range"]):
    tau = dict(zip(g.property, g.tau1))
    print(f"{center}, {rng_name}: all five tau within tolerance {bool(g.tau_within.all())}; amplitude and frequency > 0.4 s {tau['amplitude'] > 0.4 and tau['frequency'] > 0.4}; AM, FM, higher-order < 0.3 s {all(tau[k] < 0.3 for k in ['AM', 'FM', 'higher-order'])}")

**Result: partially reproduced.** With each trial's mean removed the time constants are 1.36 s (amplitude), 0.77 s (frequency), 0.10 s (AM), 0.03 s (FM) and 0.09 s (higher-order), against 0.66, 0.97, 0.11, 0.15, 0.20; only AM is within tolerance. Amplitude and frequency both exceed 0.4 s and AM, FM and higher-order are all below 0.3 s, in both fit ranges and both centerings. Sizes 0.40, 0.51, 0.97, 0.97, 0.75 (paper 0.57, 0.54, 0.86, 0.64, 0.52). The higher-order group here pools the colored noises of a file (the pooled reading of fig8a-proportion-by-property); the pair reading adopted there was run for significance only.

## fig8a-significance-fits

> The proportion of probes with significant context dependence decays fastest and is smallest for higher-order changes, with fitted (size, time constant) of (0.23, 0.25 s) for amplitude, (0.19, 0.27 s) for frequency, (0.13, 0.17 s) for AM, (0.07, 0.15 s) for FM and (0.04, 0.13 s) for higher-order properties.

As fig3a-significance-decay: one exponential with the constant fixed at the property's mean noise floor, fitted from 0 to 4 s to the proportion over the probes still running at t (and over all probes as a variant), sizes free or at least 0.

In [ ]:
exp8a = {"amplitude": (0.23, 0.25, 2.8e-3), "frequency": (0.19, 0.27, 2.3e-3), "AM": (0.13, 0.17, 2.3e-3), "FM": (0.07, 0.15, 2.1e-3), "higher-order": (0.04, 0.13, 3.6e-3)}
tab8a = []
for pr in PROPS:
    g = [s for s in syn4 if s["prop"] == pr]
    for running in [True, False]:
        pc, _ = A.population_curve([s["sig"].astype(float) for s in g], L, running)
        nz, _ = A.population_curve([np.repeat(s["noise"], 10) for s in g], L, running)
        kap = float(np.nanmean(nz[t01 <= 4]))
        kap6 = float(np.nanmean(nz))
        tt, yy = t01[::10], pc[::10]
        m = (tt <= 4) & np.isfinite(yy)
        for pos in [False, True]:
            f = A.fit_exp(tt[m], yy[m], k=1, kappa=kap, positive=pos)
            tab8a.append(dict(property=pr, denominator="running" if running else "all", sizes=">=0" if pos else "free", alpha1=f["alpha"][0], tau1=f["tau"][0], noise_floor=kap, noise_floor_0_6s=kap6,
                              paper=exp8a[pr], within=abs(f["alpha"][0] - exp8a[pr][0]) <= 0.1 * exp8a[pr][0] and abs(f["tau"][0] - exp8a[pr][1]) <= 0.1 * exp8a[pr][1]))
tab8a = pd.DataFrame(tab8a)
print(tab8a.round(4).to_string(index=False))
for (den, pos), g in tab8a.groupby(["denominator", "sizes"]):
    a = dict(zip(g.property, g.alpha1))
    print(f"{den}, sizes {pos}: all ten within {bool(g.within.all())} ({int(g.within.sum())} of 5 properties); sizes in printed order {a['amplitude'] > a['frequency'] > a['AM'] > a['FM'] > a['higher-order']}")

**Result: not reproduced.** The fitted (size, time constant) are amplitude (0.14, 0.28 s), frequency (0.16, 0.29 s), AM (0.31, 0.17 s), FM (0.05, 0.14 s) and higher-order (0.02, 0.77 s), against (0.23, 0.25), (0.19, 0.27), (0.13, 0.17), (0.07, 0.15) and (0.04, 0.13); no property has both within tolerance, and the sizes are not in the printed order (AM is largest). The noise floors are 3.4, 2.4, 4.8, 6.3 and 7.1 x 10^-3 over 0 to 4 s, and 2.7 to 3.5, 1.9 to 2.6, 4.2 to 5.0, 5.6 to 6.3 and 5.1 to 7.1 x 10^-3 over 0 to 6.07 s depending on the denominator (paper 2.8, 2.3, 2.3, 2.1 and 3.6 x 10^-3); the AM and FM floors are about two and three times the legend values under both windows.

## fig8c-relative-effects

> Changes in amplitude and frequency of the conditioning stimuli produced context effects as large and long-lasting as natural sound contexts, while AM, FM and higher-order changes produced substantially smaller and shorter effects; A1 is more sensitive to lower-order than to higher-order properties.

The area under Q(t) from 0 to 4 s for each property (fig8b-decay-by-property, each trial's mean removed, probes still running at t), divided by the area for natural sounds (fig3b-fractional-power-decay main reading). 95% intervals from 200 resamplings of 1000 samples: (probes) probes drawn with replacement, natural-sound probes resampled alike; (trials) trials drawn with replacement within each probe and context, 200 resamplings of every probe.

In [ ]:
def area(q, tmax=4.0):
    m = q["t"] <= tmax
    return float(np.nansum(q["Q"][m]) * 1e-3)
def q_from_terms(P, I):
    pr, _ = A.population_curve(P, LQ, True); iv, _ = A.population_curve(I, LQ, True)
    sm = A.running_02t(np.nan_to_num(pr), 1e-3)
    with np.errstate(invalid="ignore", divide="ignore"):
        return dict(t=np.arange(LQ) * 1e-3 + 5e-4, Q=(pr - iv) / sm)
groups8 = {"natural": nat, **{pr: [s for s in syn4 if s["prop"] == pr] for pr in PROPS}}
areas = {k: area(q_from_terms([p["pred_trial"] for p in g], [p["inv_trial"] for p in g])) for k, g in groups8.items()}
rel8 = {k: areas[k] / areas["natural"] for k in PROPS}
print("areas 0-4 s:", {k: round(v, 3) for k, v in areas.items()})
print("relative to natural sounds:", {k: round(v, 3) for k, v in rel8.items()})
def boot_probes(s, nboot=200, nsamp=1000):
    rg = np.random.default_rng(s)
    out = {k: [] for k in PROPS}
    for b in range(nboot):
        a = {}
        for k, g in groups8.items():
            idx = rg.integers(0, len(g), nsamp)
            a[k] = area(q_from_terms([g[i]["pred_trial"] for i in idx], [g[i]["inv_trial"] for i in idx]))
        for k in PROPS:
            out[k].append(a[k] / a["natural"])
    return {k: np.percentile(v, [2.5, 97.5]) for k, v in out.items()}
def boot_trials(s, nboot=200, corrected=True):
    """Trials drawn with replacement within each probe and context. corrected=True drops from Eq. 17
    the pairs of draws that repeat one trial (A.power_resampled); False applies Eq. 17 to the draws."""
    rg = np.random.default_rng(s)
    out = {k: [] for k in PROPS}
    for b in range(nboot):
        a = {}
        for k, g in groups8.items():
            P, I = [], []
            for p in g:
                X = p["X1"]; n, rep, T = X.shape
                idx = rg.integers(0, rep, (n, rep))
                while corrected and rep > 1 and any(len(set(r)) < 2 for r in idx):
                    idx = rg.integers(0, rep, (n, rep))   # Eq. 17 needs two different trials per context
                if corrected:
                    pr_, iv_ = A.power_resampled(X, idx)
                else:
                    pr_, iv_ = A.power_from_x1(X[np.arange(n)[:, None], idx], "trial")
                P.append(pr_); I.append(iv_)
            a[k] = area(q_from_terms(P, I))
        for k in PROPS:
            out[k].append(a[k] / a["natural"])
    return {k: np.percentile(v, [2.5, 97.5]) for k, v in out.items()}
def judge(ci, pt):
    lo_ok = all(ci[k][1] >= 1 for k in ["amplitude", "frequency"])
    hi_ok = all(ci[k][1] < 1 and ci[k][1] < min(ci["amplitude"][0], ci["frequency"][0]) for k in ["AM", "FM", "higher-order"])
    return lo_ok and hi_ok, None
t_b = time.time()
ci_p = {k: boot_probes(seed("fig8c-relative-effects", k)) for k in [0, 1000, 2000, 3000, 4000]}
print("probe resampling done in %.0f s" % (time.time() - t_b)); t_b = time.time()
ci_tc = {k: boot_trials(seed("fig8c-relative-effects", 1) + k) for k in [0, 1000, 2000, 3000, 4000]}
ci_t = boot_trials(seed("fig8c-relative-effects", 1), corrected=False)
print("trial resampling (second generator of the claim; corrected under five seeds, uncorrected under one) done in %.0f s" % (time.time() - t_b))
for name, ci in [("probes, claim seed", ci_p[0]), ("trials, corrected, claim seed", ci_tc[0]), ("trials, uncorrected (biased)", ci_t)]:
    print(name, {k: (round(rel8[k], 3), np.round(ci[k], 3).tolist()) for k in PROPS}, "| pattern holds:", judge(ci, rel8)[0],
          "| intervals containing the point estimate:", sum(ci[k][0] <= rel8[k] <= ci[k][1] for k in PROPS), "of 5")
for k in [1000, 2000, 3000, 4000]:
    print(f"probes, seed +{k}: pattern holds {judge(ci_p[k], rel8)[0]}; intervals", {kk: np.round(v, 2).tolist() for kk, v in ci_p[k].items()})
for k in [1000, 2000, 3000, 4000]:
    print(f"trials corrected, seed +{k}: pattern holds {judge(ci_tc[k], rel8)[0]}; intervals", {kk: np.round(v, 2).tolist() for kk, v in ci_tc[k].items()})
pt_pattern = all(rel8[k] < 1 and rel8[k] < min(rel8["amplitude"], rel8["frequency"]) for k in ["AM", "FM", "higher-order"])
print("point estimates: AM, FM, higher-order below 1 and below amplitude and frequency:", pt_pattern, "| amplitude and frequency at or above 1:", [rel8[k] >= 1 for k in ["amplitude", "frequency"]])

**Result: not reproduced, resting on an assumption.** Relative to natural sounds, the areas are 0.67 (amplitude), 0.58 (frequency), 0.02 (AM), 0.04 (FM) and 0.15 (higher-order). AM, FM and higher-order lie below 1 and below amplitude and frequency, but amplitude and frequency are well below 1, so the point estimates do not fall in the pattern the claims file names. With probes resampled the 95% intervals of amplitude (0.59 to 0.78) and frequency (0.51 to 0.66) lie below 1 under all five seeds. With trials resampled and Eq. 17 taken over pairs of different trials only (`A.power_resampled`, which removes the bias of duplicated draws), the intervals are amplitude 0.38 to 0.95 and frequency 0.42 to 0.75, also below 1 under all five seeds (upper bounds 0.91 to 0.95 and 0.72 to 0.81), and all five contain their point estimates. With Eq. 17 applied to the duplicated draws as they are, the intervals (amplitude 1.07 to 1.23, frequency 1.02 to 1.15, AM 0.60 to 0.78) contain none of the five point estimates and meet the pattern; that reading is set aside as biased (the assumption).

## fig8-natural-synthetic-probes-alike

> Natural and synthetic probes gave no difference in the context effects of the synthetic conditioning stimuli, so they were combined.

The test fixed by the claims file: per property, the proportion of significant probes between natural and synthetic probes, Fisher's exact test at 0.05, Bonferroni over the properties tested with both probe types.

In [ ]:
tabns = []
for pr in PROPS:
    g = [s for s in syn4 if s["prop"] == pr]
    a = sum(1 for s in g if s["natural"] and s["sig"].any()); b = sum(1 for s in g if s["natural"]) - a
    c = sum(1 for s in g if not s["natural"] and s["sig"].any()); d = sum(1 for s in g if not s["natural"]) - c
    if (a + b) and (c + d):
        orr, p = stats.fisher_exact([[a, b], [c, d]])
    else:
        orr, p = np.nan, np.nan
    tabns.append(dict(property=pr, natural=f"{a}/{a + b}", synthetic=f"{c}/{c + d}", nat_frac=a / (a + b) if a + b else np.nan, syn_frac=c / (c + d) if c + d else np.nan, odds_ratio=orr, p=p))
tabns = pd.DataFrame(tabns)
ntest = int(tabns.p.notna().sum())
tabns["p_bonferroni"] = np.minimum(tabns.p * ntest, 1)
print(tabns.round(4).to_string(index=False))
print("properties tested with both probe types:", ntest, "| any difference after Bonferroni at 0.05:", bool((tabns.p_bonferroni < 0.05).any()))
pp, nat_of = collections.defaultdict(bool), {}
for s in R["syn_alt"]:
    if s["reading"] == "pair" and s["rep"] >= 4:
        pp[(s["file"], s["probe"])] |= bool(s["sig"].any()); nat_of[(s["file"], s["probe"])] = s["natural"]
a = sum(1 for k, v in pp.items() if nat_of[k] and v); b = sum(1 for k in pp if nat_of[k]) - a
c = sum(1 for k, v in pp.items() if not nat_of[k] and v); d_ = sum(1 for k in pp if not nat_of[k]) - c
orr, p = stats.fisher_exact([[a, b], [c, d_]])
print(f"variant, higher-order under the pair reading of fig8a-proportion-by-property: natural {a}/{a + b} against synthetic {c}/{c + d_}, odds ratio {orr:.2f}, p = {p:.3g}, Bonferroni (x {ntest}) {min(p * ntest, 1):.3g}")


**Result: reproduced, resting on the claims file's test.** Natural against synthetic probes: amplitude 11/20 against 45/75 (p = 0.80), frequency 12/21 against 50/89 (p = 1.0), AM 19/25 against 8/14 (p = 0.29), higher-order 39/61 against 49/76 (p = 1.0); FM has no synthetic probe. No difference after Bonferroni over the 4 properties. Under the pair reading of higher-order adopted for fig8a-proportion-by-property, natural 29/58 against synthetic 35/73 (odds ratio 1.09, p = 0.86), no difference either.

## fig9-mgb-fast-decay

> In the auditory thalamus (MGB), context dependence of spike responses decayed within a short time (time constant about 80 ms; 93 probes in 14 cells), so the long-lasting component in A1 arises mainly in cortex.

PSTHs in 100-ms bins for k = 0 to 60 per context, SD over contexts (n - 1, Eq. 42), averaged over probes (each bin over the probes still running), one exponential with a free constant. Readings: the time of a bin at its start or its center; spikes from the automatic detector of `GetSpikes.m` or with half its threshold; silence counted among the contexts or not.

In [ ]:
tab9 = []
for det, sil in [("auto", True), ("auto", False), ("half", True)]:
    g = [q for q in R["mgb"] if q["detector"] == det and q["silence"] == sil and q["reps"] >= 10]
    S = np.array([q["sd"] for q in g])
    msd = np.nanmean(S, axis=0)
    for tname, off in [("bin start", 0.0), ("bin center", 0.05)]:
        tb = np.arange(61) * 0.1 + off
        m = np.isfinite(msd)
        f = A.fit_exp(tb[m], msd[m], k=1, kappa=None)
        plateau = float(np.nanmean(msd[tb >= 0.5]))
        ok = (abs(f["tau"][0] - 0.08) <= 0.008 or f["tau"][0] < 0.1) and abs(plateau - f["kappa"]) <= 0.25 * f["kappa"]
        tab9.append(dict(detector=det, silence=sil, probes=len(g), cells=len({q['cell'] for q in g}), time=tname, alpha1=f["alpha"][0], tau1=f["tau"][0], kappa=f["kappa"],
                         first_bin=msd[0], second_bin=msd[1], mean_after_0p5s=plateau, reproduced_rule=ok, not_reproduced_rule=f["tau"][0] > 0.3))
tab9 = pd.DataFrame(tab9)
print(tab9.round(3).to_string(index=False))
g = [q for q in R["mgb"] if q["detector"] == "auto" and q["silence"] and q["reps"] >= 10]
msd = np.nanmean(np.array([q["sd"] for q in g]), axis=0)
print("mean SD of PSTHs over contexts (spike/s), bins 0-9:", np.round(msd[:10], 2).tolist(), "| range over 61 bins %.2f to %.2f" % (np.nanmin(msd), np.nanmax(msd)))
for c, gg in pd.DataFrame([dict(cell=q["cell"], b0=q["sd"][0], late=np.nanmean(q["sd"][5:])) for q in g]).groupby("cell"):
    pass
pc = pd.DataFrame([dict(cell=q["cell"], b0=q["sd"][0], late=np.nanmean(q["sd"][5:])) for q in g]).groupby("cell").mean()
print("per cell, mean SD in the first bin against bins 5 and later (spike/s):")
print(pc.round(2).to_string())

**Result: reproduced.** The mean SD of PSTHs over contexts falls from 6.45 spike/s in the first bin to 2.04 in the second and stays near 1.3; one exponential gives tau = 0.055 s (below 0.1 s) with a constant of 1.27 spike/s, and the mean after 0.5 s (1.27) is within 25% of it. The same holds without silence (tau 0.067 s) and with half the spike threshold (0.061 s). The sizes differ from the paper's: alpha 5.2 (bin start) or 12.9 (bin center) against 15.0, and the constant 1.27 against 2.0 spike/s.

## fig9-example-first-bin

> In an example MGB unit, the response to a probe differed between contexts only in the first 100-ms bin of the PSTH.

Among MGB probes with the design of Figure 9A (the "7 long stimuli" sequence of Figure 2, three contexts: silence and two fragments, at least 10 repeats), reproduced when at least one has its SD of PSTHs over contexts in the first bin at least twice that of every later bin.

In [ ]:
def first_bin_only(q):
    sd = q["sd"][np.isfinite(q["sd"])]
    return bool(sd[0] > 0 and np.all(sd[0] >= 2 * sd[1:]))
g_all = [q for q in R["mgb"] if q["detector"] == "auto" and q["silence"] and q["reps"] >= 10]
g7 = [q for q in g_all if "7 long" in q["desc"] and len(q["contexts"]) == 3 and 0 in q["contexts"]]
for q in g7:
    sd = q["sd"][np.isfinite(q["sd"])]
    print(q["cell"], "probe", q["probe"], "contexts", q["contexts"], "reps", q["reps"], "SD first bin %.2f, largest later bin %.2f" % (sd[0], sd[1:].max()), "->", first_bin_only(q))
print("probes of the Figure 9A design meeting the rule:", sum(first_bin_only(q) for q in g7), "of", len(g7))
print("all MGB probes meeting the rule:", sum(first_bin_only(q) for q in g_all), "of", len(g_all))

**Result: reproduced.** Of the 12 MGB probes with the Figure 9A design (three files, contexts silence and two fragments, 10 or 17 repeats), the 4 probes of unit 20071010-ha2-003 have an SD over contexts in the first bin at least twice every later bin (26.2 to 31.4 against at most 14.7 spike/s); 35 of all 93 probes meet the rule.

## fig7-example-frequency-context

> In an example A1 neuron, a natural sound probe preceded by three DMR contexts that differed only in frequency band gave significant context dependence for about 4 s.

The claims file asks for a DMR frequency-variant cell with contexts in 0.625-2.5, 2.5-10 and 10-40 kHz and a natural sound probe. The cell below lists the DMR frequency-variant files and their bands.

In [ ]:
dmr_rows = []
for s in [s for s in syn if s["kind"] == "dmr" and s["prop"] == "frequency"]:
    rr = runs(s["sig"])
    dmr_rows.append(dict(file=s["file"].split("/")[-1], probe=s["probe"], natural_probe=s["natural"], contexts=len(s["contexts"]), rep=s["rep"], last_significant_s=round(rr[-1][1], 3) if rr else None))
print(pd.DataFrame(dmr_rows).to_string(index=False))
bands = collections.Counter()
for f in sorted({s["file"] for s in syn if s["kind"] == "dmr" and s["prop"] == "frequency"}):
    rec = A.load(f)
    b = sorted({str(np.asarray(st["param"].get("ff", "")).tolist()) if "ff" in st["param"] else st["description"].split(",")[0] for st in rec.stim if not A.is_natural(st)})
    print(f.split("/")[-1], "| natural fragment in file:", any(A.is_natural(st) for st in rec.stim), "| DMR bands:", b)
    del rec
nat_dmr = [r for r in dmr_rows if r["natural_probe"]]
print("natural probes after DMR frequency variants with a significant period ending at or after 3.6 s:", sum(1 for r in nat_dmr if r["last_significant_s"] and r["last_significant_s"] >= 3.6), "of", len(nat_dmr))
E3 = F[F.exp == 3]
print("Experiment 3 (A1, variety of stimuli): files", len(E3), "| cells", E3.cell.nunique())
with pd.option_context("display.max_colwidth", 60, "display.width", 250):
    print(E3.assign(file=E3.file.str.split("/").str[-1], desc=E3.desc.str[:60])[["file", "desc", "trig_types", "nseq", "n_sounds", "back_to_back", "min_gap_s"]].round(2).to_string(index=False))
print("experiment-3 files with sequences in their notes:", int((E3.nseq > 0).sum()), "| sounds following the previous sound within 50 ms:", int(E3.back_to_back.sum()),
      "| smallest gap between two sounds %.2f s" % E3.min_gap_s.min())

**Result: blocked.** No file has a natural-sound probe after DMR contexts in 0.625-2.5, 2.5-10 and 10-40 kHz. Experiment 3 (27 files, 12 cells) holds no sequences: none of its files lists one, no sound follows another within 50 ms, and the smallest gap between two sounds is 0.54 s (1.18 s in the files with natural sounds), so it holds no context-probe pairs and cannot hold the example. As the weaker check, of the 5 natural probes after 2-octave DMR frequency variants, 2 have a significant period ending at or after 3.6 s (4.58 and 3.82 s).

## figs1-silence-probe

> A subset of cells showed significant context dependence soon after the onset of the intersequence silences, but the response power attributable to the preceding stimulus alone faded within about 1 s after the stimulus ended.

The silence after each sequence is the probe (2.5 s, as `CxtDep.m` takes it) and the sequence's last fragment its context. Significance measure as for the other probes; power: Eq. 17 averaged over the silence probes (main), Eq. 19 as a variant; reproduced when at least one cell has a significant period starting within 0.5 s of the silence onset and the power falls below 10% of its value at onset by 1.1 s.

In [ ]:
b4 = [b for b in blank if b["rep"] >= 4]
starts = [(b["cell"], runs(b["sig"])[0][0] if b["sig"].any() else None) for b in b4]
early_cells = sorted({c for c, s0 in starts if s0 is not None and s0 <= 0.5})
print("silence probes:", len(b4), "in", len({b['cell'] for b in b4}), "cells; with a significant period:", sum(1 for c, s0 in starts if s0 is not None),
      "; cells with one starting within 0.5 s:", len(early_cells))
print("first significant times (s):", sorted(round(s0, 3) for c, s0 in starts if s0 is not None))
Lb = 2500
P = [b["pred_trial"] for b in b4]; I = [b["inv_trial"] for b in b4]
pr, _ = A.population_curve(P, Lb, True); iv, _ = A.population_curve(I, Lb, True)
tb = np.arange(Lb) * 1e-3 + 5e-4
for name, y in [("Eq. 17 stimulus-related", pr), ("Eq. 19 context-dependent", pr - iv)]:
    y0 = float(np.mean(y[:10]))
    below = A.first_below(tb, y, 0.1 * y0)
    print(f"{name}: value over the first 10 ms {y0:.3f} mV^2; first time below 10% of it {below:.3f} s; mean over 0.5-1.0 s {np.mean(y[500:1000]):.3f}; 1.0-1.1 s {np.mean(y[1000:1100]):.3f}; 2.0-2.5 s {np.mean(y[2000:]):.3f}; range {y.min():.2f} to {y.max():.2f}")
y = pr
edges = np.round(np.arange(0, 2.5001, 0.1), 3)
bins = [float(np.mean(y[int(a * 1000):int(b * 1000)])) for a, b in zip(edges[:-1], edges[1:])]
print("Eq. 17 over the silence probes in 100-ms bins from 0 s (mV^2):", np.round(bins, 2).tolist())
ref = float(np.mean(y[:10])); ipk = int(np.argmax(y))
print("peak %.2f mV^2 at %.3f s; value over the first 10 ms %.2f; first time after the peak below 10%% of the peak %.3f s" % (y[ipk], tb[ipk], ref, A.first_below(tb[ipk:], y[ipk:], 0.1 * y[ipk])))
late = [b for a, b in zip(edges[:-1], bins) if a >= 0.7]
print("100-ms bins from 0.7 to 2.5 s below 10%% of the first-10-ms value: %d of %d; below 10%% of the peak: %d of %d; mean of these bins %.2f mV^2 (%.0f%% of the first-10-ms value, %.0f%% of the peak)"
      % (sum(b < 0.1 * ref for b in late), len(late), sum(b < 0.1 * y[ipk] for b in late), len(late), np.mean(late), 100 * np.mean(late) / ref, 100 * np.mean(late) / y[ipk]))

**Result: reproduced.** Of 44 silence probes (38 cells), 28 have a significant period, and in 20 cells one starts within 0.5 s of the silence onset. The stimulus-related power over the silence probes (Eq. 17) falls from 4.59 mV^2 over the first 10 ms below 10% of it at 0.69 s, before 1.1 s, as the criterion asks; the context-dependent power (Eq. 19) at 0.68 s. The fade is not clean: the curve peaks at 12.75 mV^2 at 0.16 s, and of the 18 100-ms bins from 0.7 to 2.5 s only 4 are below 10% of the first-10-ms value (16 are below 10% of the peak); their mean is 0.85 mV^2, 19% of the first-10-ms value and 7% of the peak, with a late rise to about 3 mV^2 at 2.2 to 2.4 s. The outcome rests on the first-crossing wording of the criterion.

## Blocked claims

fig5-example-conductances and fig6-conductance-decay need the voltage-clamp recordings (currents at three holding potentials). Every whole-cell file of the data set is in mV, from current clamp (data check above), and Summary.mat lists no voltage-clamp experiment; they are not in the data set. fig7-example-frequency-context needs a cell with a natural-sound probe after DMR contexts in 0.625-2.5, 2.5-10 and 10-40 kHz; the DMR frequency files hold either 2-octave bands from 0.25 to 32 kHz with a natural probe, or 625-10000, 1250-20000, 2500-40000 Hz and 625-2500 plus 10000-40000 Hz without one (printed in its section), so no file matches the example; experiment 3, the only other A1 experiment, has no sequences and no sounds played back to back.

## Summary

A failure to reproduce here may be a failure of this attempt, not of the paper: see the reasoning and the differences from the paper's method.

| claim | outcome | main numbers |
|---|---|---|
| methods-recording-counts | partially reproduced | 305 natural probes in 39 cells and 93 MGB probes in 14 cells, as the paper; 133 A1 cells meet the criterion (paper 123); TORC 14, DMR 6, MCN 26 and MHT 2 cells by synthetic probes (paper 14, 6, 27, 39); no voltage-clamp data |
| a1-mgb-firing-rates | partially reproduced | A1 0.33 and 0.46 Hz (raw detector), MGB 0.62 and 5.07 Hz; no reading puts all four means within 10% |
| fig2-example-reliability | reproduced | population 0.319 (pairs, over probes); example cell 0.611 +/- 0.075, probe 0.737 |
| fig2-example-long-context | reproduced | significant periods up to 1.60 s and from 3.97 to 4.60 s, none between |
| fig3a-proportion-significant | partially reproduced | 187 of 305 (0.613); late fraction 0.276 to 0.295 with silence, 0.198 to 0.211 without (paper 0.237) |
| fig3a-significance-decay | partially reproduced (assumption) | with sizes at least 0: 0.15 to 0.17 at 0.28 to 0.31 s and 0.05 to 0.07 at 1.1 to 1.5 s |
| fig3b-fractional-power-decay | not reproduced | alpha 0.42 to 0.46, tau 1.50 to 2.01 s (each trial's mean removed) |
| timescale-longer-than-membrane | reproduced | 1/e lags 71 ms (noise) and 77 ms (spontaneous) |
| fig4-predictability-upper-bound | partially reproduced | 1 + alpha1 0.42 to 0.55 (within tolerance in 10 of 14 fits), tau 1.60 to 2.83 s (none within) |
| fig4-strf-performance-low | not reproduced (assumption) | see `strf_wehr.ipynb` |
| fig5-example-conductances | blocked | no voltage-clamp data |
| fig6-conductance-decay | blocked | no voltage-clamp data |
| fig8a-proportion-by-property | partially reproduced (assumption) | 56/95, 62/110, 27/39, 16/25; higher-order 64/131 with each natural sound against its own noise (88/137 pooled); order holds except under pooling |
| fig8b-decay-by-property | partially reproduced | tau 1.36, 0.77, 0.10, 0.03, 0.09 s |
| fig8a-significance-fits | not reproduced | sizes 0.14, 0.16, 0.31, 0.05, 0.02 |
| fig8c-relative-effects | not reproduced (assumption) | 0.67, 0.58, 0.02, 0.04, 0.15 of natural; amplitude and frequency intervals below 1 by probes and by trials (corrected) |
| fig8-natural-synthetic-probes-alike | reproduced (assumption) | no Fisher test significant after Bonferroni |
| fig9-mgb-fast-decay | reproduced | tau 0.055 s, plateau 1.27 spike/s |
| fig9-example-first-bin | reproduced | 4 of 12 probes of the Figure 9A design (one unit) |
| fig7-example-frequency-context | blocked | no matching cell |
| figs1-silence-probe | reproduced | 20 cells significant within 0.5 s; power below 10% at 0.69 s, but late bins average 19% of the onset value |

How the analysis differs from the paper's: contexts, probes and repeats are formed as the data set's `CxtDep.m` forms them, which reproduces the paper's sample sizes (305 natural probes in 39 cells, 93 MGB probes in 14 cells); the power measures remove each trial's mean as `CxtDep.m` does (one mean per probe, which `CxtDep.m` leaves commented out, is shown as a variant that decides nothing); spikes in A1 are detected by thresholds chosen here; the property groups of the synthetic ensembles are read from each file's descriptions, parameters and notes, which reproduce the paper's totals for amplitude, frequency and higher-order (95, 110, 137) but not for AM and FM (39 and 25 against 96 and 82); for the colored noises each natural sound is tested against its own noise, as an assumption, with the pooled and per-fragment groupings shown; the trial-resampling intervals of Figure 8C drop from Eq. 17 the pairs of draws that repeat a trial; exponential fits use a grid of start values, and for Figure 3A sizes at least 0 as an assumption.

What was found about the data set: it holds Asari's current-clamp and MGB recordings with the sample sizes of the paper, the authors' MATLAB analysis code, and Wehr's recordings; it holds no voltage-clamp recordings (Figures 5 and 6), no example cell for Figure 7, and fewer synthetic AM and FM probes than Table 1: its 27 MHT files hold 14 MHT probes with at least two MHT contexts, where Table 1 lists 178 MHT probe tests. Experiment 3 holds single sounds separated by silence, no sequences. Two cells are listed twice in Summary.mat (20060314-ha1-001 and 20060320-ha3-003, in the natural and synthetic experiments). The DMR descriptions call the 2-octave frequency variants "naturalistic sound" and "natural sound (backward)". The data set's README-Results.txt names a file `crcns-ac1-asari-resluts-2.zip`, misspelled.